In [13]:
import os
os.environ['TZ'] = 'Europe/Berlin'
 
import re
 
import anndata as ad
import pandas as pd
import scanpy as sc 
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


In [17]:
BASE = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune"
OUT_DIR = f"{BASE}/results/10_integration_GBM_HIHA"
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
adata = sc.read_h5ad(f"{BASE}/data/10_integration_GBM_HIHA/gbmap_hiha_concat_harmony.h5ad")
print("Loaded:", adata.shape)

mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/09_HIHAtlas/09_03_nmf_downsampled/mp_scores_healthy.csv", index_col=0)


Loaded: (208945, 22586)


In [ ]:
THRESHOLD = 1.0

if "condition" in adata.obs.columns:
    condition_col = "condition"
elif "dataset" in adata.obs.columns:
    label_map = {"HIHAtlas": "Healthy", "GBMap": "GBM"}
    adata.obs["condition"] = adata.obs["dataset"].map(label_map)
    condition_col = "condition"
else:
    raise ValueError("Neither 'condition' nor 'dataset' found in adata.obs.")

score_cols_native = [c for c in mp_scores.columns if re.match(r"MP\d+_score$", c)]
adata_barcodes_stripped = adata.obs_names.str.replace(r"-HIHAtlas$", "", regex=True)
aligned = mp_scores[score_cols_native].reindex(adata_barcodes_stripped)
aligned.index = adata.obs_names
for col in score_cols_native:
    adata.obs[col] = aligned[col].values
 
mp5_candidates = [c for c in adata.obs.columns if re.match(r"(HIHA_)?MP5_score$", c)]
mp5_col = mp5_candidates[0]
 
celltype_col = "decoupler_level1_celltype"
tams_label = "TAMs proinflammatory"

healthy_tams = adata[
    (adata.obs[condition_col] == "Healthy") & (adata.obs[celltype_col] == tams_label)
].copy()
healthy_tams = healthy_tams[healthy_tams.obs[mp5_col].notna()].copy()
 
healthy_tams.obs["MP5_group"] = pd.Categorical(
    (healthy_tams.obs[mp5_col] >= THRESHOLD).map({True: "MP5_high", False: "MP5_low"}),
    categories=["MP5_low", "MP5_high"],
)
print("Healthy tams subset:", healthy_tams.shape)
print(healthy_tams.obs["MP5_group"].value_counts())


Healthy pDC subset: (8738, 22586)
MP5_group
MP5_high    4584
MP5_low     4154
Name: count, dtype: int64
MP5_high pDC subset: (4584, 22586)


### parse the GMT file, keep only interferon alpha/beta/gamma terms

In [9]:
def read_gmt(path):
    """Minimal GMT parser: term \\t description \\t gene1 \\t gene2 ..."""
    gene_sets = {}
    with open(path) as f:
        for line in f:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 3:
                continue
            term, _, *genes = fields
            gene_sets[term] = [g for g in genes if g]
    return gene_sets


all_gene_sets = read_gmt("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/c5.go.bp.v2026.1.Hs.symbols.gmt")
print(f"Loaded {len(all_gene_sets)} total gene sets")

IFN_PATTERN = re.compile(
    r"INTERFERON.*(ALPHA|BETA|GAMMA)|TYPE_(I|II)_INTERFERON|IFN.?(A|B|G)(?!\w)",
    re.IGNORECASE,
)
ifn_terms = {term: genes for term, genes in all_gene_sets.items() if IFN_PATTERN.search(term)}
 
print(f"\n{len(ifn_terms)} interferon alpha/beta/gamma terms matched:")
for term in sorted(ifn_terms):
    print(f"  {term}  ({len(ifn_terms[term])} genes)")
 
if not ifn_terms:
    raise ValueError(
        "No interferon alpha/beta/gamma terms matched. Print a sample of "
        "all_gene_sets.keys() and adjust IFN_PATTERN to match the actual "
        "naming convention in this GMT file."
    )


Loaded 7538 total gene sets

27 interferon alpha/beta/gamma terms matched:
  GOBP_CELLULAR_RESPONSE_TO_INTERFERON_ALPHA  (11 genes)
  GOBP_CELLULAR_RESPONSE_TO_INTERFERON_BETA  (30 genes)
  GOBP_CELLULAR_RESPONSE_TO_TYPE_II_INTERFERON  (101 genes)
  GOBP_INTERFERON_ALPHA_PRODUCTION  (32 genes)
  GOBP_INTERFERON_BETA_PRODUCTION  (64 genes)
  GOBP_NEGATIVE_REGULATION_OF_INTERFERON_ALPHA_PRODUCTION  (6 genes)
  GOBP_NEGATIVE_REGULATION_OF_INTERFERON_BETA_PRODUCTION  (17 genes)
  GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_TYPE_II_INTERFERON  (8 genes)
  GOBP_NEGATIVE_REGULATION_OF_TYPE_II_INTERFERON_PRODUCTION  (46 genes)
  GOBP_NEGATIVE_REGULATION_OF_TYPE_I_INTERFERON_MEDIATED_SIGNALING_PATHWAY  (23 genes)
  GOBP_NEGATIVE_REGULATION_OF_TYPE_I_INTERFERON_PRODUCTION  (53 genes)
  GOBP_POSITIVE_REGULATION_OF_INTERFERON_ALPHA_PRODUCTION  (26 genes)
  GOBP_POSITIVE_REGULATION_OF_INTERFERON_BETA_PRODUCTION  (43 genes)
  GOBP_POSITIVE_REGULATION_OF_RESPONSE_TO_TYPE_II_INTERFERON  (6 genes)
  GOBP_P

### score every Healthy TAMs  on each IFN pathways

In [ ]:
scored_cols = []
for term, genes in ifn_terms.items():
    genes_in_data = [g for g in genes if g in healthy_tams.var_names]
    if len(genes_in_data) < 5:
        print(f"[{term}] only {len(genes_in_data)} genes present in data - "
              f"skipping (too few for a reliable score)")
        continue
    if len(genes_in_data) < len(genes):
        print(f"[{term}] {len(genes) - len(genes_in_data)} of {len(genes)} "
              f"genes not found in var_names, using remaining {len(genes_in_data)}")
 
    score_name = f"{term}_score"
    sc.tl.score_genes(healthy_tams, gene_list=genes_in_data, score_name=score_name, random_state=42)
    scored_cols.append(score_name)
 
print(f"\nScored {len(scored_cols)} IFN pathways on {healthy_tams.n_obs} Healthy TAMs proinflammatory")


[GOBP_CELLULAR_RESPONSE_TO_INTERFERON_BETA] 2 of 30 genes not found in var_names, using remaining 28
[GOBP_CELLULAR_RESPONSE_TO_TYPE_II_INTERFERON] 4 of 101 genes not found in var_names, using remaining 97
[GOBP_INTERFERON_BETA_PRODUCTION] 1 of 64 genes not found in var_names, using remaining 63
[GOBP_NEGATIVE_REGULATION_OF_INTERFERON_BETA_PRODUCTION] 1 of 17 genes not found in var_names, using remaining 16
[GOBP_NEGATIVE_REGULATION_OF_TYPE_II_INTERFERON_PRODUCTION] 4 of 46 genes not found in var_names, using remaining 42
[GOBP_NEGATIVE_REGULATION_OF_TYPE_I_INTERFERON_MEDIATED_SIGNALING_PATHWAY] 2 of 23 genes not found in var_names, using remaining 21
[GOBP_NEGATIVE_REGULATION_OF_TYPE_I_INTERFERON_PRODUCTION] 2 of 53 genes not found in var_names, using remaining 51
[GOBP_POSITIVE_REGULATION_OF_TYPE_II_INTERFERON_PRODUCTION] 9 of 90 genes not found in var_names, using remaining 81
[GOBP_POSITIVE_REGULATION_OF_TYPE_I_INTERFERON_PRODUCTION] 1 of 79 genes not found in var_names, using rema

In [ ]:
out_df = healthy_tams.obs[[mp5_col, "MP5_group"] + scored_cols].copy()
out_df.index.name = "barcode"
out_csv = os.path.join(OUT_DIR, "10_11_ifn_pathway_scores_TAMs.csv")
out_df.reset_index().to_csv(out_csv, index=False)
print(f"Saved per-cell IFN pathway scores to {out_csv}")

Saved per-cell IFN pathway scores to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GB_HIHA/10_11_ifn_pathway_scores_pDC.csv


### Mann-Whitney U test per pathway: MP5_high scores vs MP5_low scores

In [ ]:
high_mask = out_df["MP5_group"] == "MP5_high"
low_mask = out_df["MP5_group"] == "MP5_low"
 
test_rows = []
for score_col in scored_cols:
    term = score_col[: -len("_score")]
    high_vals = out_df.loc[high_mask, score_col]
    low_vals = out_df.loc[low_mask, score_col]
    stat, pval = mannwhitneyu(high_vals, low_vals, alternative="two-sided")
    test_rows.append({
        "term": term,
        "mean_MP5_high": high_vals.mean(),
        "mean_MP5_low": low_vals.mean(),
        "mean_diff": high_vals.mean() - low_vals.mean(),
        "n_MP5_high": len(high_vals),
        "n_MP5_low": len(low_vals),
        "U_stat": stat,
        "pval": pval,
    })
 
test_df = pd.DataFrame(test_rows)
test_df["padj"] = multipletests(test_df["pval"], method="fdr_bh")[1]
test_df = test_df.sort_values("mean_diff", key=lambda s: s.abs(), ascending=False)
 
out_test_csv = os.path.join(OUT_DIR, "10_11_TAMs_IFN_mannwhitneyu_MP5.csv")
test_df.to_csv(out_test_csv, index=False)
print(f"Saved Mann-Whitney results to {out_test_csv}")
 
print("\nIFN pathway scores, MP5_high vs MP5_low (sign: positive mean_diff = higher in MP5_high):")
print(test_df.to_string(index=False))

Saved Mann-Whitney results to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_11_pDC_IFN_mannwhitneyu_MP5.csv

IFN pathway scores, MP5_high vs MP5_low (sign: positive mean_diff = higher in MP5_high):
                                                                    term  mean_MP5_high  mean_MP5_low  mean_diff  n_MP5_high  n_MP5_low     U_stat          pval          padj
                 GOBP_NEGATIVE_REGULATION_OF_INTERFERON_ALPHA_PRODUCTION       0.000046     -0.245622   0.245668        4584       4154 17080437.0  0.000000e+00  0.000000e+00
                                        GOBP_INTERFERON_ALPHA_PRODUCTION       0.188894      0.073328   0.115566        4584       4154 16483919.0  0.000000e+00  0.000000e+00
                GOBP_NEGATIVE_REGULATION_OF_TYPE_I_INTERFERON_PRODUCTION       0.167404      0.072369   0.095035        4584       4154 16123254.0  0.000000e+00  0.000000e+00
              GOBP_NEGATIVE_REGULATION_OF_RESPONSE_TO_TY

In [24]:
high_vals.max

<bound method Series.max of barcode
54b6da1e495311ea86cade558b2f8879-HIHAtlas    0.184842
6cf55d8848b511ea87c666a4322bd84a-HIHAtlas    0.167960
5494c2ee495311ea86cade558b2f8879-HIHAtlas    0.230448
94960a26862911eebdc19e04fa809629-HIHAtlas    0.173809
6d0ceaee01e011ec97d1be2f1bab6958-HIHAtlas    0.144911
                                               ...   
06f53ef2608d11ed83fdb2a084f8f1d9-HIHAtlas    0.159399
06f9bd38608d11ed83fdb2a084f8f1d9-HIHAtlas    0.190271
06fe486c608d11ed83fdb2a084f8f1d9-HIHAtlas    0.103581
06ff8f2e608d11ed83fdb2a084f8f1d9-HIHAtlas    0.101388
07064936608d11ed83fdb2a084f8f1d9-HIHAtlas    0.200919
Name: GOBP_TYPE_I_INTERFERON_PRODUCTION_score, Length: 4584, dtype: float64>